# 01 — `Enum`, `NamedTuple`, collections avancés

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

- définir un `Enum`, `IntEnum`, `StrEnum`, `Flag`
- utiliser un `NamedTuple` typé
- maîtriser `Counter`, `defaultdict`, `deque`, `ChainMap`

## Prérequis — ce que vous connaissez déjà

À ce stade de la formation intermédiaire, vous maîtrisez :

- modèle objet complet
- type hints, dataclasses, pydantic

Ce que nous n'avons **pas encore vu** (et que nous n'utiliserons donc pas dans ce notebook) :

- décorateurs custom (jour 3)
- itertools et functools approfondis (jour 3)

## Plan

1. `Enum` : le besoin et la forme de base
2. `IntEnum`, `StrEnum`, `Flag`
3. `NamedTuple` typé
4. `collections.Counter`
5. `collections.defaultdict`
6. `collections.deque`
7. `collections.ChainMap`
8. Synthèse
9. Exercices

---

## 1. `Enum` : le besoin et la forme de base

Un `Enum` remplace les « magic strings » et les constantes flottantes par un ensemble fini, auto-documenté et vérifiable.

In [ ]:
from enum import Enum


class Statut(Enum):
    BROUILLON = 'brouillon'
    PUBLIE = 'publie'
    ARCHIVE = 'archive'


In [ ]:
Statut.PUBLIE


In [ ]:
Statut.PUBLIE.value


In [ ]:
Statut.PUBLIE.name


In [ ]:
Statut('publie')  # recherche par valeur


In [ ]:
Statut['PUBLIE']  # recherche par nom


In [ ]:
list(Statut)


---

## 2. `IntEnum`, `StrEnum`, `Flag`

Trois variantes spécialisées.

In [ ]:
from enum import IntEnum, StrEnum, Flag, auto


class Priorite(IntEnum):
    BASSE = 1
    MOYENNE = 2
    HAUTE = 3


In [ ]:
Priorite.HAUTE > Priorite.BASSE


### `StrEnum` (Python 3.11+) : chaque membre est une `str`

In [ ]:
class Niveau(StrEnum):
    INFO = 'info'
    WARN = 'warn'
    ERROR = 'error'


In [ ]:
Niveau.ERROR.upper()


In [ ]:
'info' == Niveau.INFO


### `Flag` : combinaison bit-à-bit

In [ ]:
class Permission(Flag):
    LIRE = auto()
    ECRIRE = auto()
    EXECUTER = auto()
    TOUT = LIRE | ECRIRE | EXECUTER


In [ ]:
p = Permission.LIRE | Permission.ECRIRE


In [ ]:
Permission.LIRE in p


In [ ]:
Permission.EXECUTER in p


---

## 3. `NamedTuple` typé

Un tuple avec des noms. Léger, immuable, performant. Idéal pour retourner plusieurs valeurs d'une fonction ou décrire un enregistrement en lecture seule.

In [ ]:
from typing import NamedTuple


class Coord(NamedTuple):
    x: float
    y: float
    z: float = 0.0


In [ ]:
c = Coord(1, 2)


In [ ]:
c.x, c.y, c.z


In [ ]:
tuple(c)


In [ ]:
c._replace(z=3)  # renvoie une copie modifiée


### NamedTuple vs dataclass frozen

- `NamedTuple` est un **vrai tuple** → peut être utilisé comme clé de dict, indexé par position, itéré, déstructuré.
- `dataclass(frozen=True)` est plus riche (`__post_init__`, héritage simple).
- Préférez `NamedTuple` quand la **dimension tuple** est utile.

---

## 4. `collections.Counter`

Un dict spécialisé pour compter.

In [ ]:
from collections import Counter

mots = 'le chat est sur le tapis et le chien est sous le tapis'.split()
c = Counter(mots)
c


In [ ]:
c.most_common(3)


In [ ]:
c['le']


In [ ]:
c['autre']  # pas d'erreur, renvoie 0


In [ ]:
c.update(['chat', 'chien', 'chien'])


In [ ]:
c


---

## 5. `collections.defaultdict`

Un dict qui crée automatiquement une valeur par défaut lors de l'accès à une clé absente.

In [ ]:
from collections import defaultdict

groupes = defaultdict(list)
for mot in ['pomme', 'poire', 'prune', 'abricot']:
    groupes[mot[0]].append(mot)
dict(groupes)


---

## 6. `collections.deque`

Une file double, avec des `append`/`pop` en **O(1)** aux deux bouts. Idéale pour les files et les piles.

In [ ]:
from collections import deque

q: deque[int] = deque()
q.append(1); q.append(2); q.append(3)
q.appendleft(0)
q


In [ ]:
q.popleft()


In [ ]:
q.pop()


### Rolling window avec `maxlen`

In [ ]:
fenetre: deque[int] = deque(maxlen=3)
for x in [1, 2, 3, 4, 5]:
    fenetre.append(x)
    print(list(fenetre))


---

## 7. `collections.ChainMap`

Empile plusieurs dicts pour une recherche en cascade. Très pratique pour fusionner config par défaut + overrides.

In [ ]:
from collections import ChainMap

defaut = {'host': 'localhost', 'port': 5432, 'debug': False}
override = {'debug': True}
config = ChainMap(override, defaut)
config['debug'], config['host']


---

## Synthèse

| Outil | Rôle |
|---|---|
| `Enum` / `IntEnum` / `StrEnum` | Ensemble fini de valeurs nommées |
| `Flag` | Combinaisons bit-à-bit |
| `NamedTuple` | Tuple avec noms, immuable, léger |
| `Counter` | Compter les occurrences |
| `defaultdict` | Valeur par défaut automatique |
| `deque` | File/pile à append/pop O(1) aux deux bouts |
| `ChainMap` | Recherche en cascade sur plusieurs dicts |


### Règles à retenir

1. **`Enum` pour un set fini de valeurs** ; c'est plus clair qu'une chaîne libre.
2. **`StrEnum` pour les valeurs qui circulent en JSON** : comparaison directe avec `str`.
3. **`Counter` pour comptage**, `defaultdict` pour agrégation, `deque` pour file/pile, `ChainMap` pour config.
4. **`NamedTuple` > tuple brut** dès que vous avez 2+ composants significatifs.

---

## Exercices

Les exercices sont gradués. Tous utilisent des fonctions typées (PEP 604).

### Exercice 1 — Enum de statut *(facile)*

Définir `StatutCommande(Enum)` avec `EN_ATTENTE`, `PAYEE`, `LIVREE`, `ANNULEE`. Écrire une fonction `peut_annuler(s: StatutCommande) -> bool` qui renvoie True sauf pour LIVREE.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Enum_namedtuple_collections", exercice=1)


<details>
<summary>📖 Voir la correction</summary>

```python
from enum import Enum

class StatutCommande(Enum):
    EN_ATTENTE = 'en_attente'
    PAYEE = 'payee'
    LIVREE = 'livree'
    ANNULEE = 'annulee'

def peut_annuler(s: StatutCommande) -> bool:
    return s is not StatutCommande.LIVREE

print(peut_annuler(StatutCommande.EN_ATTENTE))
print(peut_annuler(StatutCommande.LIVREE))
```

</details>

### Exercice 2 — Compter des mots *(moyen)*

Écrire une fonction `top_mots(texte: str, n: int = 5) -> list[tuple[str, int]]` qui renvoie les `n` mots les plus fréquents (casse ignorée).

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Enum_namedtuple_collections", exercice=2)


<details>
<summary>📖 Voir la correction</summary>

```python
from collections import Counter

def top_mots(texte: str, n: int = 5) -> list[tuple[str, int]]:
    return Counter(texte.lower().split()).most_common(n)

print(top_mots('le chat et le chien et le poisson et le chat', 3))
```

</details>

### Exercice 3 — Groupage avec `defaultdict` *(moyen)*

Écrire `grouper_par_categorie(items: list[tuple[str, str]]) -> dict[str, list[str]]` où chaque tuple est `(nom, categorie)`.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Enum_namedtuple_collections", exercice=3)


<details>
<summary>📖 Voir la correction</summary>

```python
from collections import defaultdict

def grouper_par_categorie(items: list[tuple[str, str]]) -> dict[str, list[str]]:
    out: defaultdict[str, list[str]] = defaultdict(list)
    for nom, cat in items:
        out[cat].append(nom)
    return dict(out)

print(grouper_par_categorie([('pain', 'alim'), ('vis', 'brico'), ('lait', 'alim')]))
```

</details>

### Exercice 4 — Flag de permissions *(difficile)*

Définir `Permission(Flag)` avec `LIRE`, `ECRIRE`, `EXECUTER`. Écrire une fonction `peut(perm: Permission, action: Permission) -> bool`. Tester avec `Permission.LIRE | Permission.ECRIRE` et vérifier les trois actions.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Enum_namedtuple_collections", exercice=4)


<details>
<summary>📖 Voir la correction</summary>

```python
from enum import Flag, auto

class Permission(Flag):
    LIRE = auto()
    ECRIRE = auto()
    EXECUTER = auto()

def peut(perm: Permission, action: Permission) -> bool:
    return action in perm

p = Permission.LIRE | Permission.ECRIRE
for a in (Permission.LIRE, Permission.ECRIRE, Permission.EXECUTER):
    print(a.name, peut(p, a))
```

</details>

---

## Ressources externes

### Documentation officielle
- [`enum`](https://docs.python.org/3/library/enum.html)
- [`collections`](https://docs.python.org/3/library/collections.html)
- [`typing.NamedTuple`](https://docs.python.org/3/library/typing.html#typing.NamedTuple)